In [40]:
# 환경 설정
from inspect import FrameInfo
from google.colab import drive
drive.mount('/content/drive')

import os
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.font_manager as fm
import pandas as pd
import numpy as np
import seaborn as sns
import requests
import xml.etree.ElementTree as ET
import pprint
import warnings
warnings.filterwarnings('ignore')

font_path = '/content/drive/MyDrive/kwukdt/seoul-bicycle-analysis/data/fonts/NanumGothic.ttf'
fm.fontManager.addfont(font_path)
mpl.rc('font', family = 'NanumGothic')
plt.rcParams['axes.unicode_minus'] = False

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [41]:
# 당일 데이터: 실시간으로 추가 중 => 전날 데이터(가장 최신 업데이트) 수집
yesterday = datetime.now() - timedelta(days=1)
yesterday_str = yesterday.strftime('%Y-%m-%d')

In [56]:
all_rows = []  # 수집한 데이터를 담을 빈 리스트
start = 1
end = 1000  # 한 번에 가져올 데이터 양 (서울시 따릉이 API는 한 번의 호출로 최대 1,000건까지 제한적으로 데이터 가져오기 가능)

while True:
    url = f'http://openapi.seoul.go.kr:8088/{api_key}/json/tbCycleRentData/{start}/{end}/{yesterday_str}/1/'

    try:
        response = requests.get(url, timeout=10).json()

        # 데이터가 정상적으로 들어있는지 확인
        if 'rentData' in response and 'row' in response['rentData']:
            rows = response['rentData']['row']
            all_rows.extend(rows)  # 가져온 데이터를 리스트에 추가

            print(f"총 {len(all_rows)}건의 데이터")

            # 다음 1000건을 위해 번호 업데이트
            start += 1000
            end += 1000
        else:
            # 더 이상 가져올 데이터가 없으면 반복문 종료
            break

    except Exception as e:
        print(f"오류 발생: {e}")
        break

# 최종 데이터프레임 생성
bike_rent_df = pd.DataFrame(all_rows)
bike_rent_df

총 968건의 데이터


,BIKE_ID,RENT_DT,RENT_ID,RENT_NM,RENT_HOLD,RTN_DT,RTN_ID,RTN_NM,RTN_HOLD,USE_MIN,USE_DST,USR_CLS_CD,SEX_CD,BIRTH_YEAR,RENT_STATION_ID,RETURN_STATION_ID,BIKE_SE_CD,START_INDEX,END_INDEX,RNUM
0,SPB-40689,2026-03-30 01:00:13,02703,서울도시가스 앞,0,2026-03-30 01:01:32,01169,염창역 1번 출구,0,1,176.32,USR_001,M,2003,ST-2018,ST-1506,일반자전거,0,0,1
1,SPB-50048,2026-03-30 01:03:17,02085,강남중학교 앞,0,2026-03-30 01:03:50,02085,강남중학교 앞,0,0,0.00,USR_001,M,1995,ST-2358,ST-2358,일반자전거,0,0,2
2,SPB-47567,2026-03-30 01:00:36,00230,영등포구청역 7번출구,0,2026-03-30 01:05:08,03221,서울특별시 남부교육지원청,0,4,562.06,USR_001,M,2007,ST-413,ST-1992,일반자전거,0,0,3
3,SPB-52701,2026-03-30 01:00:10,01103,방화역 4번출구앞,0,2026-03-30 01:05:17,05053,신마곡벽산블루밍메트로오피스텔앞,0,5,776.02,USR_001,M,1969,ST-831,ST-2883,일반자전거,0,0,4
4,SPB-72070,2026-03-30 01:01:17,00213,KT앞,0,2026-03-30 01:05:20,00219,롯데캐슬엠파이어 옆,0,4,703.19,USR_001,M,1993,ST-59,ST-62,일반자전거,0,0,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
963,SPB-63567,2026-03-30 01:01:40,00714,한국SGI 양천문화회관 앞,0,2026-03-30 04:26:04,NaN,NaN,NaN,20,3789.00,USR_001,F,1992,ST-321,NaN,일반자전거,0,0,964
964,SPB-68893,2026-03-30 01:18:18,00141,연대 대운동장 옆,0,2026-03-30 05:21:42,03009,홍대입구역 사거리,0,243,3025.73,USR_001,M,2003,ST-42,ST-2167,일반자전거,0,0,965
965,SPB-60972,2026-03-30 01:49:30,01337,돈암성당 옆,0,2026-03-30 08:34:39,00453,종로오가 지하쇼핑센터 14번출구,0,405,3088.75,USR_001,M,2008,ST-1201,ST-1606,일반자전거,0,0,966
966,SPB-69681,2026-03-30 01:09:28,03509,세종사이버대학교,0,2026-03-30 08:36:03,03579,광진 캠퍼스시티,0,447,1785.34,USR_001,NaN,1998,ST-1194,ST-2318,일반자전거,0,0,967


In [47]:
bike_rent_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 968 entries, 0 to 967
Data columns (total 20 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   BIKE_ID            968 non-null    object
 1   RENT_DT            968 non-null    object
 2   RENT_ID            968 non-null    object
 3   RENT_NM            968 non-null    object
 4   RENT_HOLD          968 non-null    object
 5   RTN_DT             968 non-null    object
 6   RTN_ID             962 non-null    object
 7   RTN_NM             962 non-null    object
 8   RTN_HOLD           962 non-null    object
 9   USE_MIN            968 non-null    object
 10  USE_DST            968 non-null    object
 11  USR_CLS_CD         968 non-null    object
 12  SEX_CD             774 non-null    object
 13  BIRTH_YEAR         939 non-null    object
 14  RENT_STATION_ID    968 non-null    object
 15  RETURN_STATION_ID  962 non-null    object
 16  BIKE_SE_CD         968 non-null    object
 1

In [48]:
BASE = '/content/drive/MyDrive/kwukdt/seoul-bicycle-analysis/data/raw/'
DATA_PATH = BASE + 'seoul_bike_rent_history.csv'

# 디렉토리가 없으면 생성
os.makedirs(os.path.dirname(DATA_PATH), exist_ok=True)

bike_rent_df.to_csv(DATA_PATH, index=False, encoding='utf-8-sig')

In [49]:
# 컬럼명 매핑용 딕셔너리
bike_rent_columns = {
    'BIKE_ID': '자전거ID',
    'RENT_DT': '대여일시',
    'RENT_ID': '대여소번호',
    'RENT_NM': '대여소명',
    'RENT_HOLD': '대여거치대',
    'RTN_DT': '반납일시',
    'RTN_ID': '반납대여소번호',
    'RTN_NM': '반납대여소명',
    'RTN_HOLD': '반납거치대',
    'USE_MIN': '사용시간(분)',
    'USE_DST': '이용거리(M)',
    'USR_CLS_CD': '사용자종류',
    'SEX_CD': '성별',
    'BIRTH_YEAR': '생년',
    'RENT_STATION_ID': '대여대여소ID',
    'RETURN_STATION_ID': '반납대여소ID',
    'BIKE_SE_CD': '자전거구분'
}

---------------------------------------------------------------------------------------------------------------------

In [60]:
api_key = '706b77416573796534366a566d426a'
all_location_data = []
start = 1
end = 1000 # 한 번에 가져올 데이터 양 (서울시 따릉이 API는 한 번의 호출로 최대 1,000건까지 제한적으로 데이터 가져오기 가능)
list_total_count = None

while True:
    url = f'http://openapi.seoul.go.kr:8088/{api_key}/xml/bikeStationMaster/{start}/{end}/'

    response = requests.get(url, timeout=10)
    root = ET.fromstring(response.text)

    if list_total_count is None:
        count_element = root.find('list_total_count')
        list_total_count = count_element.text if count_element is not None else '알 수 없음'
        print(f'총 {list_total_count}건의 데이터')

    # 현재 페이지에서 row 데이터들 찾기
    rows = root.findall('row')

    # 더 이상 가져올 데이터(row)가 없으면 반복 중단
    if not rows:
        break

    # 데이터 파싱하여 리스트에 담기
    for row in rows:
        row_dict = {child.tag: child.text for child in row}
        all_location_data.append(row_dict)

    # 다음 1000건을 위해 번호 이동
    start += 1000
    end += 1000

# 데이터프레임 변환
bike_location_df = pd.DataFrame(all_location_data)
bike_location_df

총 3411건의 데이터


,RNTLS_ID,ADDR1,ADDR2,LAT,LOT
0,ST-10,서울특별시 마포구 양화로 93,427,37.552746,126.918617
1,ST-100,서울특별시 광진구 아차산로 262,더샵스타시티 C동 앞,37.536667,127.073593
2,ST-1000,서울특별시 양천구 신정동 236,서부식자재마트 건너편,37.510380,126.866798
3,ST-1001,서울특별시 양천구 남부순환로4길20,서서울호수공원,0.000000,0.000000
4,ST-1002,서울특별시 양천구 목동동로 316-6,서울시 도로환경관리센터,37.529900,126.876541
...,...,...,...,...,...
3406,ST-995,서울특별시 양천구 중앙로 153 공중화장실,None,37.510597,126.857323
3407,ST-996,서울특별시 양천구 남부순환로88길5-16,양강중학교앞 교차로,37.524334,126.850548
3408,ST-997,서울특별시 양천구 목동중앙로 49,목동3단지 시내버스정류장,37.534390,126.869598
3409,ST-998,서울특별시 양천구 목동서로 130,목동아파트 4단지 상가동,0.000000,0.000000


In [53]:
bike_location_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3411 entries, 0 to 3410
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   RNTLS_ID  3411 non-null   object
 1   ADDR1     3411 non-null   object
 2   ADDR2     1427 non-null   object
 3   LAT       3411 non-null   object
 4   LOT       3411 non-null   object
dtypes: object(5)
memory usage: 133.4+ KB


In [54]:
BASE = '/content/drive/MyDrive/kwukdt/seoul-bicycle-analysis/data/raw/'
DATA_PATH = BASE + 'seoul_bike_location_history.csv'

# 디렉토리가 없으면 생성
os.makedirs(os.path.dirname(DATA_PATH), exist_ok=True)

bike_location_df.to_csv(DATA_PATH, index=False, encoding='utf-8-sig')

In [55]:
# 컬럼명 매핑용 딕셔너리
bike_location_columns = {
    'RNTLS_ID': '대여소_ID',
    'ADDR1': '주소1',
    'ADDR2': '주소2',
    'LAT': '위도',
    'LOT': '경도'
}